# Seven examples

**Data 110 · Fall 2026**
Rebin Muhammad

This is the code behind the seven example talks. Run each cell with **Shift+Enter**, from the top.

## How to open this in Google Colab

1. Go to [colab.research.google.com](https://colab.research.google.com).
2. Choose **File → Upload notebook** and select this file.
3. Upload these five files before you run the data cells. Click the **folder icon** on the left, then the **upload icon**. The names must stay the same. Colab forgets uploaded files when the session ends.

- `penguins.csv`
- `tempnormals.csv`
- `happiness.csv`
- `deaths.csv`
- `pumps.csv`

The planet numbers are written in the notebook, so that example needs no file.


## Setup

pandas holds the tables. matplotlib draws the charts. numpy holds a list of numbers for the density curve.


In [ ]:
import pandas as pd  # pandas holds tables. We call it pd.
import matplotlib.pyplot as plt  # matplotlib draws the chart.
import numpy as np  # numpy holds a grid of numbers for the density curve.


## Example 1 · Gentoo bills and Adelie bills

One row is one penguin. Bill length and bill depth are in millimeters. Adelie is blue, Chinstrap is green, and Gentoo is gold.


In [ ]:
penguins = pd.read_csv("penguins.csv")
colors = {"Adelie": "#0072B2", "Chinstrap": "#009E73", "Gentoo": "#E69F00"}

fig, ax = plt.subplots(figsize=(8, 4.6))
for species, color in colors.items():
    rows = penguins.loc[penguins["species"] == species]
    ax.scatter(rows["bill_length_mm"], rows["bill_depth_mm"], s=22, color=color, label=species)
ax.set_xlabel("bill length (mm)")
ax.set_ylabel("bill depth (mm)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()


## Example 2 · How wide is Jupiter?

Each number is the equatorial radius, in kilometers. Earth's radius is the unit. `round(1)` keeps one decimal place. Pluto is a dwarf planet, so it is not in this table. `barh` draws horizontal bars, and the first row sits at the bottom.


In [ ]:
planets = pd.DataFrame({
    "planet": ["Mercury", "Venus", "Earth", "Mars", "Jupiter", "Saturn", "Uranus", "Neptune"],
    "radius_km": [2439.7, 6051.8, 6378.1, 3396.2, 71492, 60268, 25559, 24764],
})
earth = planets.loc[planets["planet"] == "Earth", "radius_km"].iloc[0]
planets["earths"] = (planets["radius_km"] / earth).round(1)
planets


In [ ]:
ordered = planets.sort_values("earths")
colors = ["#E69F00" if name == "Jupiter" else "#0072B2" for name in ordered["planet"]]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(ordered["planet"], ordered["earths"], color=colors)
ax.set_xlabel("width, in Earths")
ax.set_xlim(0, 13)
for name, width in zip(ordered["planet"], ordered["earths"]):
    ax.text(width + 0.15, name, f"{width:.1f}", va="center")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()


## Example 3 · Death Valley and Houston

Each row of the file is one day. `groupby` collects the days for one city and one month, and `mean` averages them into one number. A bar below zero means Houston is warmer that month.


In [ ]:
temps = pd.read_csv("tempnormals.csv")
monthly = temps.groupby(["location", "month"], as_index=False).temperature.mean()
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
death_valley = monthly.loc[monthly["location"] == "Death Valley"].sort_values("month")
houston = monthly.loc[monthly["location"] == "Houston"].sort_values("month")
x = np.arange(12)

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(x, death_valley["temperature"], color="#0072B2", marker="o", label="Death Valley")
ax.plot(x, houston["temperature"], color="#E69F00", marker="o", label="Houston")
ax.set_xticks(x)
ax.set_xticklabels(months)
ax.set_ylabel("Temperature (°F)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()


In [ ]:
difference = death_valley["temperature"].to_numpy() - houston["temperature"].to_numpy()
colors = ["#0072B2" if value >= 0 else "#E69F00" for value in difference]

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.axhline(0, color="#1c2430", lw=0.8)
ax.bar(x, difference, color=colors)
ax.set_xticks(x)
ax.set_xticklabels(months)
ax.set_ylabel("Death Valley minus Houston (°F)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()


## Example 4 · Where do the scores sit?

One row is one country. Each histogram bar is half a point wide. The density curve uses a Gaussian kernel with bandwidth 0.35, the same setting as the slide. The gold line marks Denmark, 7.69.


In [ ]:
happiness = pd.read_csv("happiness.csv")
score = happiness["Happiness Score"]

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.hist(score, bins=np.arange(3.5, 8.01, 0.5), color="#0072B2", edgecolor="white")
ax.set_xlabel("Happiness score")
ax.set_ylabel("Countries")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()


In [ ]:
from sklearn.neighbors import KernelDensity  # KernelDensity turns the scores into a smooth curve.

grid = np.linspace(3.2, 8.2, 400)
kde = KernelDensity(kernel="gaussian", bandwidth=0.35).fit(score.to_numpy()[:, None])
density = np.exp(kde.score_samples(grid[:, None]))

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.fill_between(grid, density, color="#0072B2", alpha=0.18)
ax.plot(grid, density, color="#0072B2")
ax.axvline(7.693, color="#E69F00")
ax.text(7.61, density.max() * 0.92, "Denmark", ha="right", va="top")
ax.set_xlabel("Happiness score")
ax.set_ylabel("Density")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()


## Example 5 · Does money move with the score?

Three countries have no log GDP. `dropna` leaves them off. A blank is not a zero. The bar shows six countries, not the whole table. Hong Kong's name in the file is longer than the label on the chart.


In [ ]:
gdp = happiness.dropna(subset=["Log GDP per capita"])

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.scatter(gdp["Log GDP per capita"], gdp["Happiness Score"], s=28, color="#0072B2")
ax.set_xlabel("Log GDP per capita")
ax.set_ylabel("Happiness score")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()


In [ ]:
keep = ["Botswana", "Hong Kong S.A.R. of China", "Qatar", "United States", "Costa Rica", "Denmark"]
picked = happiness.set_index("country").loc[keep, "Happiness Score"]
labels = ["Hong Kong" if name.startswith("Hong Kong") else name for name in picked.index]

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(labels, picked, color="#0072B2")
ax.set_xlabel("Happiness score")
ax.set_xlim(0, 9.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()


## Example 6 · Two halves

`median` is the middle log GDP. Countries at or below it are the lower half. Countries above it are the higher half. Blue is Lower GDP. Gold is Higher GDP. The violin uses the same two groups.


In [ ]:
middle = gdp["Log GDP per capita"].median()
lower = gdp.loc[gdp["Log GDP per capita"] <= middle, "Happiness Score"]
higher = gdp.loc[gdp["Log GDP per capita"] > middle, "Happiness Score"]

fig, ax = plt.subplots(figsize=(8, 4.6))
box = ax.boxplot([lower, higher], tick_labels=["Lower GDP", "Higher GDP"], patch_artist=True)
box["boxes"][0].set_facecolor("#0072B2")
box["boxes"][1].set_facecolor("#E69F00")
ax.set_ylabel("Happiness score")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
violin = ax.violinplot([lower, higher], positions=[1, 2], showmedians=True)
violin["bodies"][0].set_facecolor("#0072B2")
violin["bodies"][1].set_facecolor("#E69F00")
ax.set_xticks([1, 2])
ax.set_xticklabels(["Lower GDP", "Higher GDP"])
ax.set_ylabel("Happiness score")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()


## Example 7 · The Broad Street pump

Each row of `deaths.csv` is one death. Each row of `pumps.csv` is one public water pump. The file has no pump names. The pump closest to the most deaths is the Broad Street pump. Closest means a straight line on this map, not the streets a person would walk. `idxmin` is the row of the smallest distance.


In [ ]:
deaths = pd.read_csv("deaths.csv")
pumps = pd.read_csv("pumps.csv")

nearest = []
for x, y in zip(deaths.X, deaths.Y):
    distance = ((pumps.X - x) ** 2 + (pumps.Y - y) ** 2) ** 0.5
    nearest.append(distance.idxmin())

counts = pd.Series(nearest).value_counts()
broad = counts.idxmax()
counts.loc[broad]


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6.4))
ax.scatter(deaths.X, deaths.Y, s=11, color="#0072B2", alpha=0.45)
others = pumps.drop(index=broad)
ax.scatter(others.X, others.Y, s=42, marker="s", color="#5c6b78")
ax.scatter(pumps.X.loc[broad], pumps.Y.loc[broad], s=110, color="#E69F00")
ax.annotate("Broad Street", xy=(pumps.X.loc[broad], pumps.Y.loc[broad]), xytext=(9.1, 14.15))
ax.set_aspect("equal")
ax.set_xlabel("West to east, on the map")
ax.set_ylabel("South to north, on the map")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()


In [ ]:
order = counts.reindex(range(len(pumps)), fill_value=0).sort_values()
labels = ["Broad Street" if pump == broad else "" for pump in order.index]
colors = ["#E69F00" if pump == broad else "#c5ced6" for pump in order.index]

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(range(len(order)), order.to_numpy(), color=colors)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(labels)
ax.set_xlabel("Deaths closest to this pump")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()
